# H₂ simulation: single vibrational mode

Cleaned and organized for reproducible execution from the repository root.


## H₂ 参考电子哈密顿量的 DAQS 构建（使用 OpenFermion + QuTiP）

In [ ]:
from openfermion.ops import FermionOperator
from openfermion.transforms import bravyi_kitaev
from qutip import Qobj, tensor, basis, sesolve, expect, qeye, destroy
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ===== 1. 手动构造 H₂ 分子哈密顿量（sto-3g 基组, R=0.74Å） =====
fermionic_ham = FermionOperator()
fermionic_ham += FermionOperator('', -0.81261)
fermionic_ham += FermionOperator('0^ 0', -1.25248)
fermionic_ham += FermionOperator('1^ 1', -0.47595)
fermionic_ham += FermionOperator('2^ 2', -0.47595)
fermionic_ham += FermionOperator('3^ 3', -1.25248)
fermionic_ham += FermionOperator('0^ 1^ 1 0', 0.67449)
fermionic_ham += FermionOperator('0^ 2^ 2 0', 0.66458)
fermionic_ham += FermionOperator('0^ 3^ 3 0', 0.18129)
fermionic_ham += FermionOperator('1^ 2^ 2 1', 0.18129)
fermionic_ham += FermionOperator('1^ 3^ 3 1', 0.66458)
fermionic_ham += FermionOperator('2^ 3^ 3 2', 0.67449)
fermionic_ham += FermionOperator('0^ 1^ 2 3', 0.66347)
fermionic_ham += FermionOperator('3^ 2^ 1 0', 0.66347)


In [ ]:
# ===== 2. BK 映射为泡利哈密顿量 =====
qubit_ham = bravyi_kitaev(fermionic_ham)
n_qubits = 4

def qubit_operator_to_qutip(qubit_op, n_qubits):
    pauli_map = {'X': Qobj([[0, 1], [1, 0]]),
                 'Y': Qobj([[0, -1j], [1j, 0]]),
                 'Z': Qobj([[1, 0], [0, -1]]),
                 'I': Qobj([[1, 0], [0, 1]])}
    H = 0
    for term, coeff in qubit_op.terms.items():
        ops = [pauli_map['I']] * n_qubits
        for idx, op in term:
            ops[idx] = pauli_map[op]
        H += coeff * tensor(ops)
    return H

H_el = qubit_operator_to_qutip(qubit_ham, n_qubits)

In [ ]:
# ===== 3. 构建振动模式（简谐振子） =====
n_vib = 10
a = destroy(n_vib)
w = 1.0
H_vib = w * a.dag() * a

In [ ]:
# ===== 4. 构建耦合项：Z0 ⊗ (a + a†) =====
g = 0.5
Z = Qobj([[1, 0], [0, -1]])
Z_list = [Z] + [qeye(2) for _ in range(n_qubits - 1)]
Z0 = tensor(Z_list)
a_op = a + a.dag()

# 正确指定 dims（用于避免张量维度错误）
dims = [[2] * n_qubits + [n_vib], [2] * n_qubits + [n_vib]]
Z0_ext = Qobj(tensor(Z0, qeye(n_vib)), dims=dims)
a_ext = Qobj(tensor(qeye(2 ** n_qubits), a_op), dims=dims)

H_int = g * Z0_ext * a_ext



In [ ]:
# ===== 5. 构造总哈密顿量 H_tot =====
H_el_ext = tensor(H_el, qeye(n_vib))
H_vib_ext = tensor(*[qeye(2) for _ in range(n_qubits)], H_vib)
H_tot = H_el_ext + H_vib_ext + H_int

In [ ]:
# ===== 6. 初始态：电子参考计算基态 × 振动真空态 =====
psi_el = tensor([basis(2, 0) for _ in range(n_qubits)])
psi_vib = basis(n_vib, 0)
psi0 = tensor(psi_el, psi_vib)

tlist = np.linspace(0, 25, 300)
result = sesolve(H_tot, psi0, tlist)

In [ ]:
# ===== 7. 观测比特 0 的 Z 期望值 =====
sz0 = tensor(Z0, qeye(n_vib))
z_expect = expect(sz0, result.states)

plt.plot(tlist, z_expect)
plt.xlabel("Time")
plt.ylabel("⟨Z₀⟩")
plt.title("H₂ Reference Hamiltonian + Vibronic Mode")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 输出电子本征能级 =====
e_vals = H_el.eigenenergies()
print("Lowest 4 electronic eigenenergies (manual H₂):")
print(np.sort(e_vals)[:4])